# Fetch ML Value Factor Fundamentals (date range)

Pulls the 18 fundamental ratios used by the ML value factor from Refinitiv,
point-in-time, for a universe over a date **FROM → TO** window.

- `FREQUENCY = "Q"` -> quarterly fiscal-period history (default; most
  point-in-time responsive). `RefinitivFundamentalsProvider` is initialised with
  `start_date=FROM_DATE` so the pull begins at the window start.
- Each returned row carries `effective_date` = the fiscal period-end it describes,
  and `StaticFundamentalsProvider` only ever returns rows with
  `effective_date <= as_of_date` -> **no forward-looking bias**.
- The DB write path (`collect_and_store_fundamentals`) upserts idempotently;
  re-running for the same window is safe.

Run with `python -m jupyter nbconvert --to notebook --execute --inplace
fetch_ml_value_fundamentals.ipynb` (kernel: `python3` in the project `.venv`).


In [1]:
# ===== PARAMETERS =====
UNIVERSE   = "SP500"        # portfolio_short_name whose constituents to load
FROM_DATE  = "2025-01-01"    # inclusive start of the fiscal-history pull
TO_DATE    = "2026-08-31"   # as_of anchor (latest period requested)
FREQUENCY  = "Q"            # "Q" quarterly history | "FY" single snapshot at TO_DATE
WRITE_DB   = True           # True: upsert into dbo.security_fundamentals; False: inspect only
SAMPLE_LIMIT = None         # e.g. 20 to smoke-test on a few names; None = full universe

print(f"Universe={UNIVERSE}  from={FROM_DATE}  to={TO_DATE}  freq={FREQUENCY}  write={WRITE_DB}")


Universe=SP500  from=2025-01-01  to=2026-08-31  freq=Q  write=True


In [2]:
# ----- imports / DB connection -----
import sys, os, warnings
import pandas as pd
warnings.filterwarnings("ignore")

# Make the project root importable (notebook lives under toolkit/notebooks/).
_current = os.getcwd()
_root = os.path.dirname(os.path.dirname(_current))
if _root not in sys.path:
    sys.path.append(_root)

from data_engineering.database import database as db
from analytics.factors import fundamentals as F
import lseg.data as ld
ld.open_session()  # idempotent; required for the Refinitiv fundamentals pull

engine, connection, conn_str, session = db.get_db_connection()
print("Database connection successful.")


Database connection successful.
Database connection successful.


In [3]:
# ----- resolve the universe of securities (mirrors load_refinitiv_fundamentals.py) -----
sec = pd.read_sql(
    f"""
    SELECT DISTINCT sm.security_id, sm.name AS symbol
    FROM dbo.security_master sm
    JOIN dbo.portfolio_holdings ph ON ph.security_id = sm.security_id
    JOIN dbo.portfolio p ON p.port_id = ph.port_id
    WHERE p.portfolio_short_name = '{UNIVERSE}'
    """,
    engine,
)
if sec.empty:
    raise SystemExit(f"No securities found for universe '{UNIVERSE}'.")
if SAMPLE_LIMIT:
    sec = sec.head(SAMPLE_LIMIT)
sec_sym = sec[["security_id", "symbol"]].copy()
print(f"Resolved {len(sec_sym)} securities for universe '{UNIVERSE}'.")


Resolved 539 securities for universe 'SP500'.


In [4]:
# ----- pull the 18 ratios as a point-in-time quarterly history (FROM -> TO) -----
provider = F.RefinitivFundamentalsProvider(
    orm_session=session, orm_engine=engine,
    frequency=FREQUENCY, start_date=FROM_DATE,
)

snapshot, history = provider.get_panel_history(
    sec_sym, TO_DATE, frequency=FREQUENCY, start_date=FROM_DATE,
)

print("snapshot type:", type(snapshot), "| history rows:", None if history is None else len(history))
if history is not None and not history.empty:
    print("history effective_date range:",
          pd.to_datetime(history["effective_date"]).min().date(), "->",
          pd.to_datetime(history["effective_date"]).max().date())
    print("distinct securities in history:", history["security_id"].nunique())
else:
    print("snapshot shape:", None if snapshot is None else snapshot.shape)


snapshot type: <class 'pandas.DataFrame'> | history rows: 4282
history effective_date range: 2019-09-30 -> 2026-06-30
distinct securities in history: 537


In [5]:
# ----- optional: store into dbo.security_fundamentals (idempotent upsert) -----
if WRITE_DB:
    provider_kwargs = {}
    if FREQUENCY == "Q":
        provider_kwargs["frequency"] = "Q"
        provider_kwargs["start_date"] = FROM_DATE
    n = F.collect_and_store_fundamentals(
        provider_name="refinitiv",
        symbols=sec_sym,
        as_of_date=TO_DATE,
        orm_session=session,
        orm_engine=engine,
        source_vendor="refinitiv",
        provider_kwargs=provider_kwargs or None,
    )
    print(f"Wrote {n} (security, metric) rows to dbo.security_fundamentals "
          f"(effective_date in [{FROM_DATE} -> {TO_DATE}]).")
else:
    print("WRITE_DB=False -> no DB write performed.")


Upserting 14086 security_fundamentals rows (non-destructive MERGE)...
Security fundamentals data successfully written (14086 rows upserted).
Wrote None (security, metric) rows to dbo.security_fundamentals (effective_date in [2025-01-01 -> 2026-08-31]).


In [6]:
# ----- inspect what landed: coverage + a sample of the stored ratios -----
if WRITE_DB:
    loaded = db.read_security_fundamentals(session, engine, metric_type=None)
    loaded = loaded[loaded["source_vendor"] == "refinitiv"]
    loaded["ed"] = pd.to_datetime(loaded["effective_date"])
    loaded = loaded[loaded["security_id"].isin(sec_sym["security_id"])]
    loaded = loaded[(loaded["ed"] >= pd.Timestamp(FROM_DATE)) &
                    (loaded["ed"] <= pd.Timestamp(TO_DATE))]
    print(f"Stored rows in window: {len(loaded)}  | securities: {loaded['security_id'].nunique()} "
          f"| distinct effective_dates: {loaded['ed'].dt.date.nunique()}")
    print("metrics covered:", sorted(loaded["metric_type"].unique())[:6], "...")
    print("\nSample (first security x 5 metrics):")
    s0 = sec_sym["security_id"].iloc[0]
    samp = loaded[loaded["security_id"] == s0][["metric_type", "metric_value", "effective_date"]].head(5)
    print(samp.to_string(index=False))


Stored rows in window: 228120  | securities: 539 | distinct effective_dates: 444
metrics covered: ['(CA-CL)/TA', 'Asset Turnover', 'Book Equity/TL', 'Debt Ratio', 'Debt/Equity', 'EBIT/TA'] ...

Sample (first security x 5 metrics):
             metric_type  metric_value effective_date
              Debt Ratio      0.263534     2025-10-31
Op. In./Interest Expense     13.196429     2025-10-31
          Asset Turnover      0.545926     2025-10-31
                    ROCE      0.159485     2025-10-31
          Book Equity/TL      1.126128     2025-10-31


In [7]:
# ----- point-in-time guarantee check (no forward bias) -----
# StaticFundamentalsProvider only returns fundamentals with effective_date <= the
# queried as_of_date. Verify: for every quarterly rebalance, no returned ratio is
# dated after that rebalance quarter-end.
from analytics.factors.fundamentals import RATIO_COLUMNS, StaticFundamentalsProvider
sp = StaticFundamentalsProvider(session, engine, source_vendor="refinitiv")
RATIO_SET = set(RATIO_COLUMNS)
_allf = db.read_security_fundamentals(session, engine, metric_type=None)
_allf = _allf[_allf["source_vendor"] == "refinitiv"]
_allf = _allf[_allf["security_id"].isin(sec_sym["security_id"])]
_allf = _allf[_allf["metric_type"].isin(RATIO_SET)].copy()
_allf["ed"] = pd.to_datetime(_allf["effective_date"])

worst = 0
for qdt in pd.date_range(FROM_DATE, TO_DATE, freq="QE"):
    _sd = pd.Timestamp(qdt)
    p = sp.get_panel(sec_sym, qdt.strftime("%Y-%m-%d"))
    nn = [(s, m) for s in p.index for m in RATIO_COLUMNS if pd.notna(p.loc[s, m])]
    if not nn:
        continue
    sub = _allf[_allf["ed"] <= _sd]
    mx = sub.groupby(["security_id", "metric_type"])["ed"].max()
    bad = [(s, m) for (s, m) in nn if (s, m) not in mx.index or mx.loc[(s, m)] > _sd]
    worst = max(worst, len(bad))
    assert not bad, f"FORWARD BIAS at {qdt.date()}: {bad[:3]}"
print("Max future-leaking fundamentals visible to scorer across all quarters (MUST be 0):", worst)
print("No-forward-bias check passed: fundamentals are strictly point-in-time.")

try:
    session.close(); connection.close()
except Exception:
    pass


Max future-leaking fundamentals visible to scorer across all quarters (MUST be 0): 0
No-forward-bias check passed: fundamentals are strictly point-in-time.


In [8]:
print('SMOKE_OK')

SMOKE_OK


In [9]:
print('WRITE_OK')

WRITE_OK
